# Query the bioprocess SQLite database

This notebook introduces tables, primary and foreign keys, filtering, joins, grouping, and parameterized SQL queries. Build the database first with `python -m src.data.build_database`.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "database" / "bioprocess.db"
connection = sqlite3.connect(database_path)
connection.execute("PRAGMA foreign_keys = ON")

## Inspect the schema

SQLite stores schema metadata in `sqlite_master`. The application uses five domain tables.

In [ ]:
pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type = 'table' ORDER BY name",
    connection,
)

## SELECT, WHERE, and parameterized queries

The `?` placeholder keeps values separate from SQL syntax. This is the correct pattern for values supplied by a user interface.

In [ ]:
batch_id = "B023"
b023 = pd.read_sql_query(
    """
    SELECT time_hr, ph, do_pct, glucose_g_l, lactate_g_l,
           viable_cell_density_million_ml, viability_pct, titer_g_l
    FROM sensor_data
    WHERE batch_id = ?
    ORDER BY time_hr
    """,
    connection,
    params=(batch_id,),
)
b023.head()

## JOIN metadata to outcomes

The shared `batch_id` key connects the experimental design in `batches` with final results in `outcomes`.

In [ ]:
batch_outcomes = pd.read_sql_query(
    """
    SELECT b.batch_id, b.cell_line, b.media_type, b.feed_strategy,
           o.final_titer_g_l, o.final_viability_pct
    FROM batches AS b
    JOIN outcomes AS o ON o.batch_id = b.batch_id
    ORDER BY b.batch_id
    """,
    connection,
)
batch_outcomes.head()

## GROUP BY

Group runs by media type to compare the number of runs and their average final titer.

In [ ]:
pd.read_sql_query(
    """
    SELECT b.media_type, COUNT(*) AS batch_count,
           AVG(o.final_titer_g_l) AS mean_final_titer_g_l
    FROM batches AS b
    JOIN outcomes AS o ON o.batch_id = b.batch_id
    GROUP BY b.media_type
    ORDER BY b.media_type
    """,
    connection,
)

In [ ]:
connection.close()